# GMM Estimation for MA(1) and AR(1) Models

This notebook estimates the parameters of MA(1) and AR(1) models using the **Generalized Method of Moments (GMM)**.

For both models, we match three moment conditions:
1. Mean of the series
2. Second moment (variance-related)
3. First-order autocovariance

The GMM objective is $Q(\theta) = g(\theta)' W g(\theta)$, minimised over the parameter vector.

In [6]:
import numpy as np
from scipy.optimize import minimize, LinearConstraint
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [7]:
Y = np.loadtxt('data/Time_Series.txt')   # shape: (T,)
T = len(Y)
print(f'Loaded {T} observations')
print(f'Sample mean: {Y.mean():.4f}, Sample std: {Y.std():.4f}')

Loaded 200 observations
Sample mean: 4.1989, Sample std: 2.9672


## MA(1) GMM

**Model:** $Y_t = \mu + \varepsilon_t + \theta\varepsilon_{t-1}$, where $\varepsilon_t \sim \text{i.i.d.}(0, \sigma^2)$.

**Moment conditions:**
$$g_1 = \bar{Y} - \mu$$
$$g_2 = \overline{Y^2} - \mu^2 - \sigma^2(1+\theta^2)$$
$$g_3 = \overline{Y_t}\;\overline{Y}_{t-1}  - \mu^2 - \sigma^2\theta$$

In [8]:
def gmm_objective_ma(params, Y, W):
    """GMM objective for MA(1): Q = g'Wg."""
    mu, theta, sigma = params
    T = len(Y)

    g1 = Y.mean() - mu
    g2 = (Y**2).mean() - mu**2 - sigma**2 * (1 + theta**2)
    g3 = (Y[:-1] * Y[1:]).mean() - mu**2 - sigma**2 * theta

    g = np.array([g1, g2, g3])
    return g @ W @ g


# --- Setup ---
W  = np.eye(3)          # identity weighting matrix (first-step GMM)
x0 = np.array([0.1, 0.1, 0.1])

# Constraint: -1 < theta < 1  =>  theta <= 1  and  -theta <= 1
# Written as A @ x <= b  with A=[0,1,0] and A=[0,-1,0], b=[1,1]
ma_constraints = LinearConstraint([[0, 1, 0], [0, -1, 0]], [-np.inf, -np.inf], [1, 1])

result_ma = minimize(
    gmm_objective_ma,
    x0,
    args=(Y, W),
    method='trust-constr',
    constraints=ma_constraints,
    options={'maxiter': int(1e6), 'gtol': 1e-8}
)

mu_hat, theta_hat, sigma_hat = result_ma.x
print(f'mu = {mu_hat:.6f}')
print(f'theta = {theta_hat:.6f}')
print(f'sigma = {sigma_hat:.6f}')
print(f'GMM objective value: {result_ma.fun:.8e}')
print(f'Converged: {result_ma.success}')

mu = 4.198848
theta = 0.044574
sigma = 2.964225
GMM objective value: 7.08838885e-12
Converged: True


## AR(1) GMM

**Model:** $Y_{t} = c + \phi Y_{t-1} + \varepsilon_{t}$, where $\varepsilon_t \sim \text{i.i.d.}(0, \sigma^2)$.

**Moment conditions** (using unconditional moments of a stationary AR(1)):
$$g_1 = \bar{Y} - \frac{c}{1-\phi}$$
$$g_2 = \overline{Y^2} - \frac{c^2}{(1-\phi)^2} - \frac{\sigma^2}{1-\phi^2}$$
$$g_3 = \overline{Y_t}\;\overline{Y}_{t-1} - \frac{c^2}{(1-\phi)^2} - \frac{\sigma^2 \phi}{1-\phi^2}$$

In [9]:
def gmm_objective_ar(params, Y, W):
    """GMM objective for AR(1): Q = g'Wg."""
    c, phi, sigma = params

    # Stationarity / invertibility guard
    if abs(phi) >= 1:
        return 1e12

    T = len(Y)
    mu_y  = c / (1 - phi)
    var_y = sigma**2 / (1 - phi**2)

    g1 = Y.mean() - mu_y
    g2 = (Y**2).mean() - mu_y**2 - var_y
    g3 = (Y[:-1] * Y[1:]).mean() - mu_y**2 - var_y * phi

    g = np.array([g1, g2, g3])
    return g @ W @ g


# --- Setup ---
W  = np.eye(3)
x0 = np.array([0.1, 0.1, 0.1])

# Constraint: -1 < phi < 1  (same linear form as MA)
ar_constraints = LinearConstraint([[0, 1, 0], [0, -1, 0]], [-np.inf, -np.inf], [1, 1])

result_ar = minimize(
    gmm_objective_ar,
    x0,
    args=(Y, W),
    method='trust-constr',
    constraints=ar_constraints,
    options={'maxiter': int(1e6), 'gtol': 1e-8}
)

c_hat, phi_hat, sigma_hat_ar = result_ar.x
print(f'c = {c_hat:.6f}')
print(f'phi = {phi_hat:.6f}')
print(f'sigma = {sigma_hat_ar:.6f}')
print(f'Implied mean: {c_hat/(1-phi_hat):.6f}')
print(f'GMM objective value: {result_ar.fun:.8e}')
print(f'Converged: {result_ar.success}')

c = 4.012086
phi = 0.044480
sigma = 2.964224
Implied mean: 4.198853
GMM objective value: 8.17433969e-12
Converged: True


## Summary

In [10]:
print('GMM ESTIMATION SUMMARY')
print(f'MA(1): mu={mu_hat:.4f}, theta={theta_hat:.4f}, sigma={sigma_hat:.4f}')
print(f'AR(1):  c={c_hat:.4f},  phi={phi_hat:.4f},   sigma={sigma_hat_ar:.4f}')

GMM ESTIMATION SUMMARY
MA(1): mu=4.1988, theta=0.0446, sigma=2.9642
AR(1):  c=4.0121,  phi=0.0445,   sigma=2.9642
